# Find unusual micro-watersheds

Use robust, multivariate comparison to find MWS profiles that differ strongly from the tehsil pattern while keeping missingness and the reason for each flag visible.

Run each cell with **Shift+Enter**. The location controls default to the active KYL tehsil when this notebook is downloaded from CoRE Stack.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"mws_layers\",\"label\":\"Micro-watersheds and Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Annual groundwater-storage change and MWS identifiers.\"},{\"id\":\"terrain_vector\",\"label\":\"Terrain Vector\",\"domain\":\"Land\",\"service\":\"WFS\",\"workspace\":\"terrain\",\"layerNameTemplate\":\"{district}_{tehsil}_cluster\",\"period\":\"Current terrain analysis\",\"description\":\"MWS-level plains, slopes, valleys, ridges, hills, and terrain cluster.\"},{\"id\":\"cropping_intensity\",\"label\":\"Cropping Intensity\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"crop_intensity\",\"layerNameTemplate\":\"{district}_{tehsil}_intensity\",\"period\":\"2017 to 2024\",\"description\":\"Annual cropping intensity and single-, double-, and triple-cropped area.\"},{\"id\":\"drought\",\"label\":\"Drought\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"drought\",\"layerNameTemplate\":\"{district}_{tehsil}_drought\",\"period\":\"2017 to 2024\",\"description\":\"Dry spells and weekly mild, moderate, and severe drought indicators.\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

state_input = widgets.Text(value=SCOPE["state"], description="State:", layout=widgets.Layout(width="98%"))
district_input = widgets.Text(value=SCOPE["district"], description="District:", layout=widgets.Layout(width="98%"))
tehsil_input = widgets.Text(value=SCOPE["tehsil"], description="Tehsil:", layout=widgets.Layout(width="98%"))
display(widgets.VBox([
    widgets.HTML("<b>Study location</b><br><small>Change a name here, then rerun the data cells. No Python editing is needed.</small>"),
    state_input, district_input, tehsil_input,
]))

def selected_scope():
    return {
        "state": state_input.value.strip(),
        "district": geoserver_name(district_input.value),
        "tehsil": geoserver_name(tehsil_input.value),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
PROFILE_METRICS = [
    "Area (ha)", "Mean annual groundwater change", "Recent net groundwater change",
    "Mean cropping intensity", "Mean moderate + severe drought weeks",
    "Plain terrain (%)", "Slope and hill terrain (%)",
]

def build_mws_profile(mws_frame, crop_frame, drought_frame, terrain_frame):
    mws = with_uid(mws_frame).set_index("uid")
    crop = with_uid(crop_frame).set_index("uid")
    drought = with_uid(drought_frame).set_index("uid")
    terrain = with_uid(terrain_frame).set_index("uid")
    common = mws.index.intersection(crop.index).intersection(drought.index).intersection(terrain.index)
    result = pd.DataFrame(index=common)
    area = pd.to_numeric(mws.get("area_in_ha"), errors="coerce").reindex(common)
    groundwater = [column for column in mws.columns if re.match(r"^\d{4}_\d{4}$", column)]
    crop_years = [column for column in crop.columns if re.match(r"^cropping_intensity_\d{4}$", column)]
    drought_years = sorted(set(re.findall(r"\d{4}", " ".join(drought.columns))))
    result["Area (ha)"] = area
    result["Mean annual groundwater change"] = component_values(mws, groundwater, "DeltaG").mean(axis=1).reindex(common)
    result["Recent net groundwater change"] = pd.to_numeric(mws.get("Net2020_25"), errors="coerce").reindex(common)
    result["Mean cropping intensity"] = numeric(crop, crop_years).mean(axis=1).reindex(common)
    drought_values = pd.DataFrame(index=drought.index)
    for year in drought_years:
        moderate = pd.to_numeric(drought.get(f"w_mod_{year}"), errors="coerce")
        severe = pd.to_numeric(drought.get(f"w_sev_{year}"), errors="coerce")
        if moderate is not None and severe is not None:
            drought_values[year] = moderate.add(severe, fill_value=np.nan)
    result["Mean moderate + severe drought weeks"] = drought_values.mean(axis=1).reindex(common)
    plains = pd.to_numeric(terrain.get("plain_area"), errors="coerce").reindex(common)
    slopes = pd.to_numeric(terrain.get("slopy_area"), errors="coerce").reindex(common)
    hills = pd.to_numeric(terrain.get("hill_slope"), errors="coerce").reindex(common)
    result["Plain terrain (%)"] = plains
    result["Slope and hill terrain (%)"] = slopes.add(hills, fill_value=np.nan)
    result.index.name = "MWS UID"
    return result.replace([np.inf, -np.inf], np.nan)

def robust_standardize(profile):
    values = profile[PROFILE_METRICS]
    center = values.median()
    mad = values.sub(center).abs().median()
    iqr = values.quantile(0.75) - values.quantile(0.25)
    scale = (1.4826 * mad).where(mad > 0, iqr / 1.349).replace(0, np.nan)
    return values.sub(center).div(scale), center, scale

def rank_outliers(profile, threshold=3.5):
    standardized, center, scale = robust_standardize(profile)
    absolute = standardized.abs()
    comparable = absolute.notna().sum(axis=1)
    score = absolute.max(axis=1, skipna=True).where(comparable >= 4)
    reason = absolute.apply(
        lambda row: row.dropna().idxmax() if not row.dropna().empty else "Insufficient data",
        axis=1,
    )
    ranked = profile.copy()
    ranked["Comparable metrics"] = comparable
    ranked["Robust outlier score"] = score
    ranked["Largest departure"] = reason
    ranked["Flagged"] = score.ge(threshold) & comparable.ge(4)
    return ranked.sort_values("Robust outlier score", ascending=False), standardized

def plot_outlier_scores(ranked, threshold=3.5):
    shown = ranked.dropna(subset=["Robust outlier score"]).head(12).sort_values("Robust outlier score")
    if shown.empty:
        print("No MWS has enough comparable metrics to plot.")
        return
    colors = ["#dc2626" if value >= threshold else "#64748b" for value in shown["Robust outlier score"]]
    ax = shown["Robust outlier score"].plot.barh(figsize=(10, 6), color=colors)
    ax.axvline(threshold, color="#991b1b", linestyle="--", label=f"Flag threshold = {threshold}")
    ax.set(title="Largest robust departure for each MWS", xlabel="Absolute robust standardized score", ylabel="MWS UID")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 1. Build a comparable MWS profile

Four relevant layers are joined by MWS UID. No missing value is replaced with a made-up average.

In [ ]:
mws_geojson = await load_geojson("mws_layers")
crop_geojson = await load_geojson("cropping_intensity")
drought_geojson = await load_geojson("drought")
terrain_geojson = await load_geojson("terrain_vector")
profile = build_mws_profile(to_frame(mws_geojson), to_frame(crop_geojson),
                            to_frame(drought_geojson), to_frame(terrain_geojson))
print(f"Built {len(profile)} MWS profiles from {len(PROFILE_METRICS)} metrics.")

## 2. Rank unusual profiles

A robust score uses each metric's median and median absolute deviation. The table states the largest departure and how many metrics were comparable.

In [ ]:
ranked, standardized = rank_outliers(profile, threshold=3.5)
display(ranked[["Robust outlier score", "Largest departure", "Comparable metrics", "Flagged"]].head(12).round(2))
plot_outlier_scores(ranked, threshold=3.5)

## 3. Highlight flagged MWSes

Red polygons are unusual relative to this tehsil and this metric set—not necessarily degraded, erroneous, or in need of the same intervention.

In [ ]:
flagged = ranked[ranked["Flagged"]]
print(f"Flagged {len(flagged)} of {len(ranked)} MWSes.")
display(flagged[PROFILE_METRICS + ["Largest departure"]].round(2))
show_on_map("outlier-mws", features_for_uids(mws_geojson, flagged.index),
            "Notebook · unusual MWS profiles", fillColor="#ef4444", strokeColor="#7f1d1d", fillOpacity=0.62)

## Interpretation

Outlier detection is a question generator. Inspect the named departure, raw attributes, geometry, measurement coverage, and local context before drawing a conclusion.

## Optional: see standardized departures

In [ ]:
display(standardized.loc[ranked.head(12).index].round(2))